In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, log_loss, confusion_matrix,
                             classification_report, roc_auc_score, roc_curve)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
import pickle as pk
import numpy as np

# EDA

In [2]:
df = pd.read_csv("diabetes.csv")
df.head()

In [3]:
df.info()

In [4]:
print(df.shape)
print(df.isnull().sum())
print(df.isnull().sum().sum() / len(df) * 100)

In [5]:
# 1. Removing null rows
print(df.shape)
df.dropna(inplace=True)
print(df.shape)

In [6]:
# 2. Removing duplicate values
print(df.duplicated().sum())
df.drop_duplicates(inplace=True, ignore_index=True)
print(df.duplicated().sum())
print(df.shape)

## Outlier Treatment

In [7]:
# 3a. Visualise outliers BEFORE treatment
plt.figure(figsize=(14, 6))
df.boxplot()
plt.title("Boxplot – Before Outlier Treatment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [8]:
# 3b. Identify outliers using IQR for each numeric column
print("Outlier counts per column (before treatment):")
for col in df.select_dtypes(include=["int", "float"]).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"  {col:<30} outliers: {n_outliers}")

In [9]:
# 3c. Replace outliers with IQR boundary values (Winsorising)
def replace_outliers(df):
    def replace(col):
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_extreme = Q1 - 1.5 * IQR
        upper_extreme = Q3 + 1.5 * IQR
        df[col] = df[col].apply(
            lambda x: lower_extreme if x < lower_extreme
            else upper_extreme if x > upper_extreme
            else x
        )
    for c in df.select_dtypes(include=["int", "float"]).columns:
        replace(c)

replace_outliers(df)

# Visualise AFTER treatment
plt.figure(figsize=(14, 6))
df.boxplot()
plt.title("Boxplot – After Outlier Treatment (Winsorising)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("Outlier treatment complete.")

In [10]:
# 4. Multi-collinearity detection – correlation heatmap
corr = df.corr()
print(corr)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [11]:
'''
Columns with multicollinearity > 0.5:  Age, Pregnancies
Columns with multicollinearity ≈ 0.5:  Insulin-SkinThickness, Glucose-Outcome
'''

In [12]:
target   = df["Outcome"]
features = df.drop(columns=["Outcome"])

In [13]:
# 5. VIF – detect multicollinearity
vif = pd.DataFrame()
vif["Features"]  = features.columns
vif["VIF_SCORES"] = [variance_inflation_factor(features.values, i)
                     for i in range(len(features.columns))]
vif.sort_values(ascending=False, by="VIF_SCORES")

In [14]:
'''
VIF > 10: BloodPressure, BMI, Glucose, Age
Dropping BloodPressure and BMI first; Glucose is clinically important so keep it.
'''
features.drop(columns=["BloodPressure", "BMI"], inplace=True)

In [15]:
vif = pd.DataFrame()
vif["Features"]  = features.columns
vif["VIF_SCORES"] = [variance_inflation_factor(features.values, i)
                     for i in range(len(features.columns))]
vif.sort_values(ascending=False, by="VIF_SCORES")
'''
After dropping BloodPressure and BMI, VIF for Glucose and Age are acceptable.
'''

In [16]:
# 6. Feature–target correlation using F-classif
f_reg = f_classif(features, target)
print(f_reg[0])
pd.Series(f_reg[0], index=features.columns).sort_values(ascending=False).plot(kind='bar')
plt.title("F-score: Feature Correlation with Target")
plt.show()

In [17]:
corrdf = pd.DataFrame()
corrdf["Features"]    = features.columns
corrdf["Correlation"] = f_classif(features, target)[0]
print(corrdf.sort_values(ascending=False, by="Correlation"))

In [18]:
features.head()

# Visualization

In [19]:
# Pairplot
sns.pairplot(features, diag_kind="kde")
plt.show()

# Logistic Regression

In [20]:
x_train, x_test, y_train, y_test = train_test_split(
    features, target, train_size=0.8, random_state=150)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

In [21]:
# Scaling
std_scaler = StandardScaler()
scale_cols  = ["Glucose", "SkinThickness", "Insulin"]
x_train[scale_cols] = std_scaler.fit_transform(x_train[scale_cols])
x_test[scale_cols]  = std_scaler.transform(x_test[scale_cols])
x_train.head()

In [22]:
# Model training
log_model = LogisticRegression()
log_model.fit(x_train, y_train)

print("Coefficients :", log_model.coef_)
print("Intercept    :", log_model.intercept_)

In [23]:
# Predictions
y_pred_test  = log_model.predict(x_test)
sigmoid_test = log_model.predict_proba(x_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_test))
print("Log Loss :", log_loss(y_test, sigmoid_test))

In [24]:
# Threshold adjustment
y_pred_adj = [1 if p >= 0.5 else 0 for p in sigmoid_test]
print("Adjusted threshold accuracy:", accuracy_score(y_test, y_pred_adj))

# Performance Metrics

In [25]:
# Confusion Matrix
conf = confusion_matrix(y_test, y_pred_test)
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [26]:
print(classification_report(y_test, y_pred_test))

In [27]:
# ROC Curve
auc_score = roc_auc_score(y_test, sigmoid_test)
fpr, tpr, _ = roc_curve(y_test, sigmoid_test)

plt.plot(fpr, tpr, lw=2, color="red", label=f"AUC Score: {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curve"); plt.legend(); plt.grid(); plt.show()

# Deployment – Pickle Files

In [28]:
# Save trained model
with open("log_reg_assignment.pkl", "wb") as f:
    pk.dump(log_model, f)
print("Model saved -> log_reg_assignment.pkl")

# Save scaler
with open("std_scaler_log_reg_assignment.pkl", "wb") as f:
    pk.dump(std_scaler, f)
print("Scaler saved -> std_scaler_log_reg_assignment.pkl")

# Save feature column names (needed by the app to match input order)
feature_cols = list(x_train.columns)
with open("feature_cols.pkl", "wb") as f:
    pk.dump(feature_cols, f)
print("Feature columns saved -> feature_cols.pkl")
print("Features used:", feature_cols)

In [29]:
# Verify pickle files work correctly
loaded_model  = pk.load(open("log_reg_assignment.pkl", "rb"))
loaded_scaler = pk.load(open("std_scaler_log_reg_assignment.pkl", "rb"))
loaded_cols   = pk.load(open("feature_cols.pkl", "rb"))

# Test prediction with a sample row from x_test
sample = x_test.iloc[[0]]
pred   = loaded_model.predict(sample)
prob   = loaded_model.predict_proba(sample)[:, 1]
print("Pickle verification:")
print("  Feature columns  :", loaded_cols)
print("  Sample prediction:", pred[0], " | Probability:", round(prob[0], 4))
print("  Actual label     :", y_test.iloc[0])
print("Pickle files are working correctly.")

# Streamlit App

In [30]:
# The cell below writes the complete Streamlit app to a file called app.py
# To run it:  open a terminal and type ->  streamlit run app.py

streamlit_code = '''
import streamlit as st
import pickle
import numpy as np
import pandas as pd

# ── Load saved artefacts ────────────────────────────────────────────────────
model       = pickle.load(open("log_reg_assignment.pkl",          "rb"))
scaler      = pickle.load(open("std_scaler_log_reg_assignment.pkl","rb"))
feature_cols = pickle.load(open("feature_cols.pkl",               "rb"))

# ── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(page_title="Diabetes Predictor", page_icon="🩺", layout="centered")
st.title("🩺 Diabetes Prediction App")
st.markdown("Enter the patient\'s details below and click **Predict** to get a diagnosis.")

st.divider()

# ── Input form ──────────────────────────────────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    pregnancies    = st.number_input("Pregnancies",           min_value=0,   max_value=20,   value=1,   step=1)
    glucose        = st.number_input("Glucose (mg/dL)",       min_value=0,   max_value=300,  value=120, step=1)
    skin_thickness = st.number_input("Skin Thickness (mm)",   min_value=0,   max_value=100,  value=20,  step=1)

with col2:
    insulin        = st.number_input("Insulin (mu U/ml)",     min_value=0,   max_value=900,  value=80,  step=1)
    dpf            = st.number_input("Diabetes Pedigree Fn",  min_value=0.0, max_value=3.0,  value=0.5, step=0.01)
    age            = st.number_input("Age (years)",           min_value=1,   max_value=120,  value=30,  step=1)

st.divider()

# ── Prediction ───────────────────────────────────────────────────────────────
if st.button("🔍 Predict", use_container_width=True):
    # Build input DataFrame matching training column order
    input_data = pd.DataFrame([[pregnancies, glucose, skin_thickness,
                                insulin, dpf, age]],
                              columns=feature_cols)

    # Scale the same columns that were scaled during training
    scale_cols = ["Glucose", "SkinThickness", "Insulin"]
    input_data[scale_cols] = scaler.transform(input_data[scale_cols])

    # Predict
    prediction  = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.divider()
    if prediction == 1:
        st.error(f"⚠️ **Diabetic**  —  Probability: {probability:.1%}")
        st.markdown("The model predicts this patient is **likely diabetic**. "
                    "Please consult a medical professional for further evaluation.")
    else:
        st.success(f"✅ **Non-Diabetic**  —  Probability of Diabetes: {probability:.1%}")
        st.markdown("The model predicts this patient is **likely not diabetic**. "
                    "Maintain a healthy lifestyle and schedule regular check-ups.")

    st.divider()
    st.subheader("Input Summary")
    display = pd.DataFrame({
        "Feature": ["Pregnancies","Glucose","Skin Thickness","Insulin",
                    "Diabetes Pedigree Function","Age"],
        "Value"  : [pregnancies, glucose, skin_thickness, insulin, dpf, age]
    })
    st.dataframe(display, use_container_width=True)
'''

with open("app.py", "w") as f:
    f.write(streamlit_code.strip())

print("app.py written successfully.")
print()
print("To launch the Streamlit app, run this command in your terminal:")
print("  streamlit run app.py")

In [31]:
# Preview the app.py contents
with open("app.py", "r") as f:
    print(f.read())

## Deployment Instructions

### Option A — Run Locally
1. Install Streamlit if not already installed:
   ```
   pip install streamlit
   ```
2. Make sure these files are in the same folder:
   - `app.py`
   - `log_reg_assignment.pkl`
   - `std_scaler_log_reg_assignment.pkl`
   - `feature_cols.pkl`
3. Open a terminal in that folder and run:
   ```
   streamlit run app.py
   ```
4. The app will open automatically at `http://localhost:8501`

### Option B — Deploy to Streamlit Cloud (Free)
1. Push the 4 files above to a **public GitHub repository**.
2. Go to [https://streamlit.io/cloud](https://streamlit.io/cloud) and sign in with GitHub.
3. Click **New app** → select your repository and `app.py`.
4. Click **Deploy** — Streamlit Cloud will build and host it for free.
5. Share the generated public URL with your instructor.

### App Features
- Accepts all 6 model input features via number inputs
- Applies the same StandardScaler used during training
- Displays prediction (Diabetic / Non-Diabetic) with probability
- Shows a clean input summary table

In [32]:
'''
1) Difference between Precision and Recall

a) Precision: Out of all predicted positives, how many are actually positive.
   Precision = TP / (TP + FP)
   Focuses on: false positives

b) Recall: Out of all actual positives, how many were correctly predicted.
   Recall = TP / (TP + FN)
   Focuses on: false negatives

In Short:
   Precision = correctness of positive predictions
   Recall    = coverage of actual positives


2) Cross-Validation

A technique where data is split into multiple folds (e.g. 5 or 10).
The model trains on some folds and validates on the remaining fold,
repeating until every sample has been tested exactly once.

Why important in binary classification:
 - Gives a more reliable estimate of model performance
 - Reduces overfitting to a single train/test split
 - Helps detect class imbalance issues across folds
 - Ensures metrics like precision/recall are stable across splits

In short: Cross-validation checks how well a binary classifier generalises to unseen data.
'''